# Oil & Gas Production Intelligence & Well Performance Analytics

**Purpose:** Analyze field production, well performance, water injection, WOR/GOR and operational efficiency using a reproducible Python workflow.

**Portfolio focus:** Data cleaning → KPI development → exploratory analysis → SQL analysis → Power BI-ready outputs → business insights.

> **Analytical note:** Relationships between injection, production and ratios are treated as associations that may require further investigation; this notebook does not claim causation.

## 1. Business Questions

This project is structured around practical Data Analyst questions:

1. What are the field-level oil, gas and water production totals?
2. How does production change over time?
3. Which wells contribute the most oil and gas?
4. Which wells show declining or weaker production performance?
5. How do water injection and production trends move over time?
6. Which wells have increasing Water-Oil Ratio (WOR) or Gas-Oil Ratio (GOR)?
7. Which wells have stronger operational availability?
8. Which wells should be prioritized for further operational investigation?

In [ ]:
# Core libraries
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_FILE = Path("oilwell_production_data.xlsx")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
DB_FILE = OUTPUT_DIR / "oil_gas_analytics.db"

print(f"Data file: {DATA_FILE.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

## 2. Load the Source Data

The analysis uses the workbook's **Daily Production Data** sheet as the primary source. Individual well sheets are retained as optional reference sheets but are not required for the analytical pipeline. This avoids maintaining multiple copies of the same calculations.

In [ ]:
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"{DATA_FILE} was not found. Put the Excel file in the same folder as this notebook."
    )

sheets = pd.read_excel(DATA_FILE, sheet_name=None)
print("Available sheets:", list(sheets.keys()))

if "Daily Production Data" not in sheets:
    raise KeyError("Expected sheet 'Daily Production Data' was not found.")

daily = sheets["Daily Production Data"].copy()
print(f"Loaded {len(daily):,} rows and {daily.shape[1]} columns.")
display(daily.head())

## 3. Data Cleaning & Quality Checks

The cleaning step standardizes column names, converts numeric measures, parses dates, removes exact duplicates, and creates a consistent `WELL_NAME` field. Calculated ratios are created later from aggregated volumes instead of averaging daily ratios.

In [ ]:
# Standardize column names
daily.columns = (
    daily.columns
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", "_", regex=True)
)

# Standardize the well-name column without assuming the workbook has already been cleaned
if "WELL_NAME" not in daily.columns and "NPD_WELL_BORE_NAME" in daily.columns:
    daily = daily.rename(columns={"NPD_WELL_BORE_NAME": "WELL_NAME"})

required = [
    "DATEPRD", "WELL_NAME", "BORE_OIL_VOL", "BORE_GAS_VOL",
    "BORE_WAT_VOL", "BORE_WI_VOL", "ON_STREAM_HRS"
]
missing = [c for c in required if c not in daily.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

daily["DATEPRD"] = pd.to_datetime(daily["DATEPRD"], errors="coerce")

numeric_cols = [
    c for c in [
        "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL",
        "BORE_WI_VOL", "ON_STREAM_HRS", "WOR"
    ] if c in daily.columns
]
for col in numeric_cols:
    daily[col] = pd.to_numeric(daily[col], errors="coerce").fillna(0)

# Remove exact duplicate records and rows without a usable date/well
before = len(daily)
daily = daily.drop_duplicates().dropna(subset=["DATEPRD", "WELL_NAME"]).copy()
duplicates_removed = before - len(daily)

# Remove negative physical volume values from production/injection measures.
# This keeps the analytical dataset conservative; unusual records should be investigated separately.
volume_cols = [c for c in ["BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL"] if c in daily.columns]
for col in volume_cols:
    daily.loc[daily[col] < 0, col] = np.nan
    daily[col] = daily[col].fillna(0)

print(f"Rows after cleaning: {len(daily):,}")
print(f"Exact duplicate rows removed: {duplicates_removed:,}")
print(f"Date range: {daily['DATEPRD'].min().date()} to {daily['DATEPRD'].max().date()}")
print(f"Unique wells: {daily['WELL_NAME'].nunique()}")

quality = pd.DataFrame({
    "metric": ["rows", "columns", "unique_wells", "missing_dates", "missing_well_names"],
    "value": [
        len(daily), daily.shape[1], daily["WELL_NAME"].nunique(),
        daily["DATEPRD"].isna().sum(), daily["WELL_NAME"].isna().sum()
    ]
})
display(quality)

In [ ]:
# Optional schema inspection
display(pd.DataFrame({"column": daily.columns, "dtype": daily.dtypes.astype(str)}))
if "WELL_TYPE" in daily.columns:
    print("\nWell types:")
    display(daily["WELL_TYPE"].value_counts(dropna=False).to_frame("records"))

## 4. Create a Consistent Analytical Dataset

The master dataset is used for all calculations. Producer/injector status is inferred from the available `WELL_TYPE` and production/injection volumes rather than manually listing well names.

In [ ]:
# Identify producer/injector records using available source information
if "WELL_TYPE" in daily.columns:
    daily["is_producer"] = daily["WELL_TYPE"].astype(str).str.upper().str.contains("OP", na=False)
else:
    daily["is_producer"] = daily["BORE_OIL_VOL"].gt(0) | daily["BORE_GAS_VOL"].gt(0)

daily["is_production_active"] = (
    daily["ON_STREAM_HRS"].gt(0) &
    (daily["BORE_OIL_VOL"].gt(0) | daily["BORE_GAS_VOL"].gt(0) | daily["BORE_WAT_VOL"].gt(0))
)

daily["is_injection_active"] = (
    daily["ON_STREAM_HRS"].gt(0) & daily["BORE_WI_VOL"].gt(0)
)

daily["is_operational"] = daily["is_production_active"] | daily["is_injection_active"]

producer_daily = daily[daily["is_producer"]].copy()
print(f"Producer records: {len(producer_daily):,}")
print(f"Producer wells: {producer_daily['WELL_NAME'].nunique()}")

## 5. Field-Level KPIs

In [ ]:
field_kpis = {
    "Total Oil Production (Sm³)": daily["BORE_OIL_VOL"].sum(),
    "Total Gas Production (Sm³)": daily["BORE_GAS_VOL"].sum(),
    "Total Water Production (Sm³)": daily["BORE_WAT_VOL"].sum(),
    "Total Water Injection (Sm³)": daily["BORE_WI_VOL"].sum(),
    "Unique Wells": daily["WELL_NAME"].nunique(),
    "Operational Records": int(daily["is_operational"].sum()),
}
field_kpi_df = pd.DataFrame(field_kpis.items(), columns=["KPI", "Value"])
display(field_kpi_df)

## 6. Monthly Field Production

Monthly aggregation reduces daily noise and provides a useful reporting view for production monitoring.

In [ ]:
monthly_field = (
    daily.set_index("DATEPRD")
    .resample("ME")[["BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL"]]
    .sum()
    .reset_index()
)

monthly_field["GOR"] = np.where(
    monthly_field["BORE_OIL_VOL"] > 0,
    monthly_field["BORE_GAS_VOL"] / monthly_field["BORE_OIL_VOL"],
    np.nan
)
monthly_field["WOR"] = np.where(
    monthly_field["BORE_OIL_VOL"] > 0,
    monthly_field["BORE_WAT_VOL"] / monthly_field["BORE_OIL_VOL"],
    np.nan
)

display(monthly_field.head())

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(monthly_field["DATEPRD"], monthly_field["BORE_OIL_VOL"], linewidth=2, label="Oil")
ax.plot(monthly_field["DATEPRD"], monthly_field["BORE_GAS_VOL"], linewidth=2, label="Gas")
ax.set_title("Monthly Field Oil and Gas Production")
ax.set_xlabel("Date")
ax.set_ylabel("Production (Sm³)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Well-Level Production Performance

In [ ]:
well_summary = (
    daily.groupby("WELL_NAME")
    .agg(
        Total_Oil_Sm3=("BORE_OIL_VOL", "sum"),
        Total_Gas_Sm3=("BORE_GAS_VOL", "sum"),
        Total_Water_Sm3=("BORE_WAT_VOL", "sum"),
        Total_Injection_Sm3=("BORE_WI_VOL", "sum"),
        Recorded_Days=("DATEPRD", "nunique"),
        Operational_Days=("is_operational", "sum"),
        Avg_On_Stream_Hours=("ON_STREAM_HRS", "mean"),
    )
    .reset_index()
)

well_summary["Operational_Time_Pct"] = np.where(
    well_summary["Recorded_Days"] > 0,
    well_summary["Operational_Days"] / well_summary["Recorded_Days"] * 100,
    np.nan
)
well_summary["WOR"] = np.where(
    well_summary["Total_Oil_Sm3"] > 0,
    well_summary["Total_Water_Sm3"] / well_summary["Total_Oil_Sm3"],
    np.nan
)
well_summary["GOR"] = np.where(
    well_summary["Total_Oil_Sm3"] > 0,
    well_summary["Total_Gas_Sm3"] / well_summary["Total_Oil_Sm3"],
    np.nan
)

well_summary = well_summary.sort_values("Total_Oil_Sm3", ascending=False).reset_index(drop=True)
display(well_summary)

In [ ]:
top_n = min(10, len(well_summary))
plot_df = well_summary.head(top_n).sort_values("Total_Oil_Sm3")

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(plot_df["WELL_NAME"], plot_df["Total_Oil_Sm3"])
ax.set_title(f"Top {top_n} Wells by Cumulative Oil Production")
ax.set_xlabel("Cumulative Oil Production (Sm³)")
ax.set_ylabel("Well")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Monthly Well Production & Decline Indicators

A simple decline indicator compares the first and last non-zero monthly production observations for each well. It is a screening metric, not a reservoir-engineering decline-curve model.

In [ ]:
monthly_well = (
    daily.groupby([pd.Grouper(key="DATEPRD", freq="ME"), "WELL_NAME"])[
        ["BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL"]
    ].sum()
    .reset_index()
)

monthly_well["WOR"] = np.where(
    monthly_well["BORE_OIL_VOL"] > 0,
    monthly_well["BORE_WAT_VOL"] / monthly_well["BORE_OIL_VOL"],
    np.nan
)
monthly_well["GOR"] = np.where(
    monthly_well["BORE_OIL_VOL"] > 0,
    monthly_well["BORE_GAS_VOL"] / monthly_well["BORE_OIL_VOL"],
    np.nan
)

def first_last_decline(group):
    positive = group.loc[group["BORE_OIL_VOL"] > 0].sort_values("DATEPRD")
    if len(positive) < 2:
        return pd.Series({"First_Month_Oil": np.nan, "Last_Month_Oil": np.nan, "Oil_Change_Pct": np.nan})
    first = positive.iloc[0]["BORE_OIL_VOL"]
    last = positive.iloc[-1]["BORE_OIL_VOL"]
    change = (last - first) / first * 100 if first else np.nan
    return pd.Series({"First_Month_Oil": first, "Last_Month_Oil": last, "Oil_Change_Pct": change})

decline_summary = monthly_well.groupby("WELL_NAME").apply(first_last_decline, include_groups=False).reset_index()
well_summary = well_summary.drop(columns=[c for c in ["First_Month_Oil", "Last_Month_Oil", "Oil_Change_Pct"] if c in well_summary.columns])
well_summary = well_summary.merge(decline_summary, on="WELL_NAME", how="left")

display(well_summary.sort_values("Oil_Change_Pct").head(10))

## 9. Water Injection & Production Relationship

The chart compares field-level monthly water injection with oil production. The purpose is monitoring and hypothesis generation, not proving causality.

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 6))
ax1.plot(monthly_field["DATEPRD"], monthly_field["BORE_OIL_VOL"], linewidth=2, label="Oil Production")
ax1.set_xlabel("Date")
ax1.set_ylabel("Oil Production (Sm³)")

ax2 = ax1.twinx()
ax2.plot(monthly_field["DATEPRD"], monthly_field["BORE_WI_VOL"], linestyle="--", linewidth=2, label="Water Injection")
ax2.set_ylabel("Water Injection (Sm³)")

ax1.set_title("Monthly Water Injection vs Oil Production")
ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax1.grid(axis="y", alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 10. WOR — Water-Oil Ratio

WOR is calculated as aggregated water production divided by aggregated oil production. Aggregating volumes before calculating the ratio avoids distortion from averaging daily ratios.

In [ ]:
producer_monthly = monthly_well[monthly_well["BORE_OIL_VOL"] > 0].copy()

wor_trend = (
    producer_monthly.groupby("DATEPRD")["WOR"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(wor_trend["DATEPRD"], wor_trend["WOR"], linewidth=2)
ax.set_title("Average Producer-Well WOR Trend")
ax.set_xlabel("Date")
ax.set_ylabel("WOR (Water / Oil)")
ax.grid(axis="y", alpha=0.3)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Identify wells with high recent WOR relative to their historical median
recent_cutoff = monthly_well["DATEPRD"].max() - pd.DateOffset(months=6)
recent_wor = (
    monthly_well[(monthly_well["DATEPRD"] >= recent_cutoff) & monthly_well["WOR"].notna()]
    .groupby("WELL_NAME")["WOR"]
    .mean()
    .rename("Recent_Avg_WOR")
)
historical_wor = (
    monthly_well[monthly_well["WOR"].notna()]
    .groupby("WELL_NAME")["WOR"]
    .median()
    .rename("Historical_Median_WOR")
)
wor_screen = pd.concat([recent_wor, historical_wor], axis=1).reset_index()
wor_screen["WOR_Change_Pct"] = np.where(
    wor_screen["Historical_Median_WOR"] > 0,
    (wor_screen["Recent_Avg_WOR"] - wor_screen["Historical_Median_WOR"]) / wor_screen["Historical_Median_WOR"] * 100,
    np.nan
)
wor_screen = wor_screen.sort_values("WOR_Change_Pct", ascending=False)
display(wor_screen.head(10))

## 11. GOR — Gas-Oil Ratio

GOR is calculated from monthly aggregated gas and oil volumes. Large changes can be flagged for further investigation.

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 6))
ax1.plot(monthly_field["DATEPRD"], monthly_field["BORE_OIL_VOL"], linewidth=2, label="Oil Production")
ax1.set_xlabel("Date")
ax1.set_ylabel("Oil Production (Sm³)")

ax2 = ax1.twinx()
ax2.plot(monthly_field["DATEPRD"], monthly_field["GOR"], linestyle="--", linewidth=2, label="Field GOR")
ax2.set_ylabel("GOR (Gas / Oil)")
ax1.set_title("Field GOR vs Oil Production")
ax1.grid(axis="y", alpha=0.3)
ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 12. Operational Performance

Operational time is measured as operational records divided by recorded days. This is a simple availability KPI based on the source data and should not be interpreted as an official uptime metric without validating the business definition.

In [ ]:
operational_summary = well_summary[[
    "WELL_NAME", "Recorded_Days", "Operational_Days",
    "Operational_Time_Pct", "Avg_On_Stream_Hours",
    "Total_Oil_Sm3", "Total_Gas_Sm3", "Total_Water_Sm3"
]].copy()

operational_summary = operational_summary.sort_values(
    "Operational_Time_Pct", ascending=False
)
display(operational_summary)

## 13. SQL Analysis

The cleaned analytical tables are stored in SQLite so that the same business questions can be answered using SQL. This demonstrates the full analyst workflow rather than relying only on Pandas.

In [ ]:
# Recreate the local SQLite database so the notebook is reproducible
if DB_FILE.exists():
    DB_FILE.unlink()

with sqlite3.connect(DB_FILE) as conn:
    well_summary.to_sql("well_summary", conn, index=False, if_exists="replace")
    monthly_well.to_sql("monthly_well", conn, index=False, if_exists="replace")
    monthly_field.to_sql("monthly_field", conn, index=False, if_exists="replace")

print(f"SQLite database created at: {DB_FILE.resolve()}")

In [ ]:
with sqlite3.connect(DB_FILE) as conn:
    top_wells_sql = pd.read_sql_query("""
        SELECT
            WELL_NAME,
            Total_Oil_Sm3,
            Total_Gas_Sm3,
            Operational_Time_Pct
        FROM well_summary
        ORDER BY Total_Oil_Sm3 DESC
        LIMIT 10;
    """, conn)

display(top_wells_sql)

In [ ]:
with sqlite3.connect(DB_FILE) as conn:
    ranked_wells_sql = pd.read_sql_query("""
        SELECT
            WELL_NAME,
            Total_Oil_Sm3,
            RANK() OVER (ORDER BY Total_Oil_Sm3 DESC) AS Oil_Production_Rank
        FROM well_summary
        ORDER BY Oil_Production_Rank;
    """, conn)

display(ranked_wells_sql)

In [ ]:
with sqlite3.connect(DB_FILE) as conn:
    monthly_sql = pd.read_sql_query("""
        SELECT
            DATEPRD,
            BORE_OIL_VOL AS Oil_Production,
            BORE_GAS_VOL AS Gas_Production,
            BORE_WI_VOL AS Water_Injection
        FROM monthly_field
        ORDER BY DATEPRD;
    """, conn)

display(monthly_sql.head(12))

## 14. Power BI-Ready Outputs

The following CSV files are generated for dashboarding. Import them into Power BI and build relationships/visuals from these stable analytical tables rather than directly from the raw Excel workbook.

In [ ]:
powerbi_field = monthly_field.copy()
powerbi_well = well_summary.copy()
powerbi_monthly_well = monthly_well.copy()

field_csv = OUTPUT_DIR / "powerbi_field_monthly.csv"
well_csv = OUTPUT_DIR / "powerbi_well_summary.csv"
monthly_well_csv = OUTPUT_DIR / "powerbi_monthly_well.csv"

powerbi_field.to_csv(field_csv, index=False)
powerbi_well.to_csv(well_csv, index=False)
powerbi_monthly_well.to_csv(monthly_well_csv, index=False)

print("Created:")
for p in [field_csv, well_csv, monthly_well_csv, DB_FILE]:
    print(f" - {p}")

## 15. Business Insight Summary

The tables below are designed to support concise management-style reporting. Values are calculated from the source data when the notebook is executed.

In [ ]:
# Generate a compact, data-driven insight summary
top_well = well_summary.iloc[0] if len(well_summary) else None
lowest_change = well_summary.dropna(subset=["Oil_Change_Pct"]).sort_values("Oil_Change_Pct").head(3)
high_wor = wor_screen.head(3)

print("KEY INSIGHTS")
print("=" * 60)
print(f"• Total field oil production: {daily['BORE_OIL_VOL'].sum():,.0f} Sm³")
print(f"• Total field gas production: {daily['BORE_GAS_VOL'].sum():,.0f} Sm³")
print(f"• Total field water production: {daily['BORE_WAT_VOL'].sum():,.0f} Sm³")
print(f"• Total water injection: {daily['BORE_WI_VOL'].sum():,.0f} Sm³")
if top_well is not None:
    print(f"• Highest cumulative oil producer: {top_well['WELL_NAME']} ({top_well['Total_Oil_Sm3']:,.0f} Sm³)")
if len(lowest_change):
    print("• Wells with the largest first-to-last monthly oil decline screening metric:")
    for _, row in lowest_change.iterrows():
        print(f"  - {row['WELL_NAME']}: {row['Oil_Change_Pct']:.1f}%")
if len(high_wor):
    print("• Wells with the largest recent-vs-historical WOR increase screening metric:")
    for _, row in high_wor.iterrows():
        print(f"  - {row['WELL_NAME']}: {row['WOR_Change_Pct']:.1f}%")
print("\nInterpretation: flagged wells should be investigated using operational context; these metrics do not establish causation.")

## 16. Recommended Power BI Dashboard

### Page 1 — Executive Overview
- Total oil, gas, water and injection KPIs
- Monthly production trend
- Top producing wells
- Production contribution by well

### Page 2 — Well Performance
- Well selector
- Oil/gas/water trends
- Cumulative production
- Operational time
- WOR and GOR

### Page 3 — Injection & Production
- Water injection vs oil production
- WOR trend
- Recent vs historical WOR
- Wells requiring further investigation

### Page 4 — Operational Reporting
- Operational time by well
- On-stream hours
- Production contribution
- Decline-screening table

**Suggested Power BI measures:** Total Oil, Total Gas, Total Water, Total Injection, Average WOR, Average GOR, Operational %, and Oil Contribution %.

## 17. Project Takeaways

- Built a reproducible **Python/Pandas data-cleaning and KPI pipeline** for oil & gas production data.
- Converted daily production data into **field-level and well-level analytical tables**.
- Used **SQL and window functions** for ranking and reporting analysis.
- Created **Power BI-ready datasets** for interactive reporting.
- Evaluated production, injection, WOR, GOR and operational performance using defensible screening metrics.
- Translated the analysis into **business questions and actionable investigation areas** rather than only producing charts.